In [1]:
%cd ..

/home/sergey/Desktop/Менторство/test-api


In [2]:
"""Ячейка 1 — импорты и переменные окружения.

`load_dotenv()` подтягивает `LLM_API_KEY` из `.env`, без него RAGAS-метрики
(Ячейка 5) не смогут поднять LLM-клиент. `json` и `Path` нужны только в
Ячейке 6 для записи итогового файла с метриками.
"""
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
"""Ячейка 2 — golden-датасет эталонных вопросов.

Структура: EN-вопросы по темам корпуса (базовое трио + расширение:
ensemble / cross_validation / preprocessing / compose / grid_search /
impute / feature_selection), RU для мультиязычного retrieval'а,
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""
GOLDEN = [
    # --- EN: базовое трио (linear_model / tree / model_evaluation) ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- EN: расширение корпуса ---
    {
        "question": "What is the idea behind bagging and Random Forest?",
        "ground_truth": "Bagging trains many base estimators on bootstrap samples and "
                        "aggregates their predictions; Random Forest additionally "
                        "samples a random subset of features at each split.",
        "ground_truth_url_keywords": ["ensemble"],
    },
    {
        "question": "How does AdaBoost differ from bagging?",
        "ground_truth": "AdaBoost trains weak learners sequentially, reweighting "
                        "misclassified samples so later estimators focus on hard "
                        "examples; bagging trains estimators independently in parallel.",
        "ground_truth_url_keywords": ["ensemble"],
    },
    {
        "question": "What is the difference between K-Fold and StratifiedKFold?",
        "ground_truth": "K-Fold splits data into k folds of roughly equal size; "
                        "StratifiedKFold preserves class proportions in each fold, "
                        "which matters for imbalanced classification.",
        "ground_truth_url_keywords": ["cross_validation"],
    },
    {
        "question": "What does cross_val_score do in scikit-learn?",
        "ground_truth": "cross_val_score evaluates an estimator by cross-validation: "
                        "it fits on training folds and scores on the held-out fold, "
                        "returning an array of scores for each split.",
        "ground_truth_url_keywords": ["cross_validation"],
    },
    {
        "question": "When should I use StandardScaler vs MinMaxScaler?",
        "ground_truth": "StandardScaler centers features to zero mean and unit variance; "
                        "MinMaxScaler maps features to a fixed range like [0, 1]. "
                        "Prefer StandardScaler for many linear models; MinMax when "
                        "a bounded range is required.",
        "ground_truth_url_keywords": ["preprocessing"],
    },
    {
        "question": "What is OneHotEncoder used for?",
        "ground_truth": "OneHotEncoder converts categorical features into binary "
                        "indicator columns, one per category, so linear models can "
                        "use categorical inputs without ordinal assumptions.",
        "ground_truth_url_keywords": ["preprocessing"],
    },
    {
        "question": "Why use Pipeline and ColumnTransformer together?",
        "ground_truth": "Pipeline chains preprocessing and an estimator so they fit "
                        "and transform consistently; ColumnTransformer applies "
                        "different transformers to different column subsets before "
                        "the final step.",
        "ground_truth_url_keywords": ["compose"],
    },
    {
        "question": "What is the difference between GridSearchCV and RandomizedSearchCV?",
        "ground_truth": "GridSearchCV exhaustively evaluates all parameter combinations "
                        "in a grid; RandomizedSearchCV samples a fixed number of "
                        "combinations from distributions, which is cheaper for large "
                        "search spaces.",
        "ground_truth_url_keywords": ["grid_search"],
    },
    {
        "question": "How does SimpleImputer fill missing values?",
        "ground_truth": "SimpleImputer replaces missing values with a statistic along "
                        "each column: mean, median, most frequent, or a constant, "
                        "depending on the strategy parameter.",
        "ground_truth_url_keywords": ["impute"],
    },
    {
        "question": "What is the difference between SelectKBest and RFE?",
        "ground_truth": "SelectKBest keeps the k features with the highest univariate "
                        "scores; RFE recursively fits an estimator and prunes the "
                        "least important features until the desired number remains.",
        "ground_truth_url_keywords": ["feature_selection"],
    },
    # --- RU (мультиязычный retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "Зачем нужен Pipeline в scikit-learn?",
        "ground_truth": "Pipeline объединяет шаги предобработки и модель в один "
                        "объект: fit/transform применяются согласованно, без утечки "
                        "данных из теста в обучение при кросс-валидации.",
        "ground_truth_url_keywords": ["compose"],
    },
    {
        "question": "Что такое GridSearchCV?",
        "ground_truth": "GridSearchCV перебирает сетку гиперпараметров с "
                        "кросс-валидацией и выбирает комбинацию с лучшим score, "
                        "затем может переобучить лучшую модель на всех данных.",
        "ground_truth_url_keywords": ["grid_search"],
    },
    # --- meta (about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по документации scikit-learn: линейные "
                        "модели, деревья, метрики, ансамбли, кросс-валидация, "
                        "препроцессинг, пайплайны, grid search, импутация и "
                        "отбор признаков.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [4]:
"""Ячейка 3 — прямое подключение к RAG-pipeline без HTTP-слоя.

`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в этом же процессе. Прямой вызов даёт чистые тайминги и доступ к
`retrieved_contexts`, которые понадобятся RAGAS в Ячейке 5.
"""
from app.config import settings
from app.rag.chain import build_rag_chain
chain, retriever = build_rag_chain()

/home/sergey/Desktop/Менторство/test-api/project-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7282.22it/s]


In [5]:
"""Ячейка 4 — Recall@k для retriever'а.

Прогоняем каждый golden-вопрос через retriever, забираем URL'ы топ-k чанков
и считаем hit по keyword-совпадению. Дополнительно копим
`retrieved_contexts` — они уйдут в RAGAS на следующем шаге.
"""
def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.

    Используется в Recall@k: keyword-matching на уровне модуля sklearn
    (`linear_model` / `tree` / `model_evaluation`) — этого достаточно
    для понимания «не промазал ли retriever мимо темы целиком».
    """
    if not expected_keywords:  # ловушка под OOC-вопросы, если решите добавлять
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 1.000  (22/22)


In [6]:
"""Ячейка 5 — generation-метрики через RAGAS.

RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`). LLM
и эмбеддер оборачиваем в `LangchainLLMWrapper` и
`LangchainEmbeddingsWrapper` — без этого `evaluate()` не поймёт, как
через них ходить. Считаем Faithfulness (опора на контекст) и
ResponseRelevancy (релевантность ответа вопросу).
"""
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

from app.llm import get_llm

# Прогоняем chain и собираем ответы
for r in results:
    r["response"] = chain.invoke(r["question"])

# RAGAS 0.2.x требует именно эти имена полей в Dataset:
# user_input / response / retrieved_contexts / reference
ragas_data = Dataset.from_list([
    {
        "user_input": r["question"],
        "response": r["response"],
        "retrieved_contexts": r["retrieved_contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

# LLM и эмбеддер для RAGAS оборачиваются в специальные wrapper'ы
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)

ragas_scores = evaluate(
    dataset=ragas_data,
    metrics=[
        Faithfulness(llm=llm),
        ResponseRelevancy(llm=llm, embeddings=emb),
    ],
)
print(ragas_scores)

Evaluating:   0%|                                                                                 | 0/44 [00:00<?, ?it/s]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x763bcc2df340> is already entered
Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 44/44 [00:24<00:00,  1.77it/s]


{'faithfulness': 0.9397, 'answer_relevancy': 0.9277}


In [7]:
"""Ячейка 6 — сохраняем сводку метрик в JSON.

Берём mean по каждой RAGAS-метрике, добавляем retriever-Recall@k и
метаданные прогона (модель, embedder, top_k). Файл `rag_metrics.json`
цитируется в README + входит в будущую легенду на собесе.
"""
# RAGAS 0.2.x возвращает EvaluationResult — берём .to_pandas() для агрегации
df = ragas_scores.to_pandas()
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}
Path("notebooks/rag_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False)
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "n_questions": 22,
  "model": "deepseek-v4-flash",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 22
  },
  "generation": {
    "faithfulness": 0.9396775473330496,
    "answer_relevancy": 0.9277224606411285
  }
}
